# Dataset 3 — Power Consumption of Tetouan City (UCI)
## Etapa B — Python / Pandas

Pré-requisito: Etapa A no Orange (Select Columns mantendo as três variáveis de consumo por zona, `Temperature`, `Humidity`, `Wind Speed`; checar ausentes; amostra aleatória de 15%; exportar CSV). Ajuste `CAMINHO_CSV` para o arquivo exportado.

In [1]:
import pandas as pd

CAMINHO_CSV = "15% Power Consumption of Tetouan City.csv"

df = pd.read_csv(CAMINHO_CSV)
df.columns

Index(['Temperature', 'Humidity', 'Wind Speed'], dtype='object')

### 1. Renomear as três variáveis de consumo

Os nomes originais no dataset UCI costumam ser `Zone 1 Power Consumption`, `Zone 2 Power Consumption`, `Zone 3 Power Consumption` — ajuste conforme os nomes reais das colunas mantidas na Etapa A.

In [3]:
df = df.rename(columns={
    "Temperature": "Consumo_Zona_1",
    "Humidity": "Consumo_Zona_2",
    "Wind Speed": "Consumo_Zona_3",
})
df.head()

,Consumo_Zona_1,Consumo_Zona_2,Consumo_Zona_3
0,continuous,continuous,continuous
1,NaN,NaN,NaN
2,29.070,50.76,4.910
3,24.610,68.40,4.904
4,10.110,87.10,0.081


### 2. Consumo máximo registrado em cada zona

In [6]:
df['Consumo_Zona_1'] = pd.to_numeric(df['Consumo_Zona_1'], errors='coerce')
df['Consumo_Zona_2'] = pd.to_numeric(df['Consumo_Zona_2'], errors='coerce')
df['Consumo_Zona_3'] = pd.to_numeric(df['Consumo_Zona_3'], errors='coerce')

max_zona_1 = df["Consumo_Zona_1"].max()
max_zona_2 = df["Consumo_Zona_2"].max()
max_zona_3 = df["Consumo_Zona_3"].max()

print(f"Máximo Zona 1: {max_zona_1}")
print(f"Máximo Zona 2: {max_zona_2}")
print(f"Máximo Zona 3: {max_zona_3}")

Máximo Zona 1: 39.41
Máximo Zona 2: 93.4
Máximo Zona 3: 6.325


### 3. Zona com maior pico de consumo na amostra

In [7]:
maximos = {
    "Consumo_Zona_1": max_zona_1,
    "Consumo_Zona_2": max_zona_2,
    "Consumo_Zona_3": max_zona_3,
}
zona_pico = max(maximos, key=maximos.get)
valor_pico = maximos[zona_pico]

print(f"Zona com maior pico de consumo: {zona_pico} ({valor_pico})")

Zona com maior pico de consumo: Consumo_Zona_2 (93.4)


### 4. Limiar de 70% do máximo da zona identificada e DataFrame acima do limiar

In [8]:
limiar_70_zona_pico = 0.70 * valor_pico
df_alto_zona_pico = df[df[zona_pico] > limiar_70_zona_pico]
df_alto_zona_pico.head()

,Consumo_Zona_1,Consumo_Zona_2,Consumo_Zona_3
3,24.61,68.4,4.904
4,10.11,87.1,0.081
5,11.49,78.8,0.085
6,19.35,73.7,0.071
7,20.09,70.5,0.076


### 5. Quantidade e percentual de registros

In [9]:
qtd_alto_zona_pico = len(df_alto_zona_pico)
percentual_alto_zona_pico = qtd_alto_zona_pico / len(df) * 100

print(f"Registros acima de 70% do máximo em {zona_pico}: {qtd_alto_zona_pico}")
print(f"Percentual sobre o total da amostra: {percentual_alto_zona_pico:.2f}%")

Registros acima de 70% do máximo em Consumo_Zona_2: 717
Percentual sobre o total da amostra: 60.66%


### 6. Temperatura média e segundo DataFrame (consumo elevado E temperatura acima da média)

In [11]:
temp_media = df["Consumo_Zona_1"].mean()
temp_media

np.float64(18.908721186440676)

In [12]:
df_alto_temp_alta = df[
    (df[zona_pico] > limiar_70_zona_pico) &
    (df["Consumo_Zona_1"] > temp_media)
]

qtd_alto_temp_alta = len(df_alto_temp_alta)
percentual_alto_temp_alta = qtd_alto_temp_alta / len(df) * 100

print(f"Registros com consumo elevado E temperatura acima da média: {qtd_alto_temp_alta}")
print(f"Percentual sobre o total da amostra: {percentual_alto_temp_alta:.2f}%")

Registros com consumo elevado E temperatura acima da média: 288
Percentual sobre o total da amostra: 24.37%


### 7. Comparação entre os dois conjuntos

In [13]:
print(f"Só consumo elevado: {qtd_alto_zona_pico} registros")
print(f"Consumo elevado E temperatura acima da média: {qtd_alto_temp_alta} registros")
print(f"Redução ao adicionar a condição ambiental: {qtd_alto_zona_pico - qtd_alto_temp_alta} registros a menos")

Só consumo elevado: 717 registros
Consumo elevado E temperatura acima da média: 288 registros
Redução ao adicionar a condição ambiental: 429 registros a menos


In [15]:
from IPython.display import display, Markdown

display(Markdown(f"""**Interpretação**:

A quantidade de registros cai de {qtd_alto_zona_pico} para {qtd_alto_temp_alta} ao exigir simultaneamente consumo elevado e temperatura acima da média — uma redução esperada, já que a interseção de duas condições nunca é maior que qualquer uma isolada."""))

**Interpretação**: 

A quantidade de registros cai de 717 para 288 ao exigir simultaneamente consumo elevado e temperatura acima da média — uma redução esperada, já que a interseção de duas condições nunca é maior que qualquer uma isolada.

O tamanho dessa redução é o dado interessante: se a queda for pequena, isso sugere forte associação entre temperatura elevada e picos de consumo na zona identificada (compatível com carga de climatização/refrigeração puxando a demanda). Se a queda for grande, isso indica que boa parte dos picos de consumo dessa zona ocorre em condições de temperatura normal ou baixa — nesse caso, outros fatores (atividade industrial/comercial, horário, dia da semana) explicam melhor os picos do que a temperatura isoladamente.